# Optional: rebuild the intermediate source features
This notebook demonstrates the engineering preceding the final v2.2 experiment. It reads bundled prepared PMFBY/DES/geography master, NASA daily observations and MODIS seasonal extracts. It does not claim to reconstruct the original API downloads or raw DES parsing.

It rebuilds the nearest-location join (Haversine, <=75 km), complete seasonal calendars, missing-day-aware dry spells, rainfall/heat features, prior-year anomalies and intermediate Gold. Historical complete-case readiness is retained **as a diagnostic**, not as eligibility for the final two-million-row experiment. Notebook 01 includes every context.

Outputs are new preparation tables. Notebook 01 deliberately uses the historical intermediate snapshots so rebuilding here cannot silently change the reported experiment.

In [ ]:
import re
from pathlib import Path
import sys
PROJECT_ROOT = None
candidate = Path(PROJECT_ROOT or Path.cwd()).resolve()
root = (
    next((p for p in (candidate,
             *candidate.parents) if (p / 'project_config.json').is_file()),
         None)
)
if root is None:
    raise FileNotFoundError('Import/clone the COMPLETE repository; set PROJECT_ROOT if needed.')
sys.path.insert(0, str(root))
from src.project_runtime import ProjectRuntime
import json
import math
from functools import reduce
from time import perf_counter
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array
runtime = ProjectRuntime(spark, root, run_kind='preparation')
VERSION = runtime.config['simulation_version']
VERSION = '2.1.0'
RUN_ID, PREFIX = (runtime.run_id, runtime.prefix)
SEED, TARGET = (runtime.config['seed'], runtime.config['synthetic_rows'])
assert TARGET >= 4497
MAX_GRID_DISTANCE_KM, MIN_DAY_SHARE = (75.0, 0.95)
TRAIN_YEARS, VALIDATION_YEAR, TEST_YEAR = ([2018, 2019, 2020], 2021, 2022)
TRAIN_SAMPLE_SHARE = 0.2
CAPACITIES = [0.1, 0.2, 0.3]
SOURCE_VERSIONS, PUBLISHED, TIMINGS = (runtime.source_versions, runtime.published, runtime.timings)
STARTED = runtime.started
publish, table = (runtime.publish, runtime.table)

def unique(df, keys, description):
    assert not df.groupBy(*keys).count().filter('count > 1').limit(1).count(), description

def require(df, columns):
    assert len(df.columns) == len(set(df.columns)), 'Duplicate columns'
    missing = set(columns) - set(df.columns)
    assert not missing, 'Missing columns: ' + repr(sorted(missing))

def uniform(id_col, salt):
    return (F.pmod(F.xxhash64(F.col(id_col), F.lit(SEED), F.lit(salt)), F.lit(1000000000)) + F.lit(0.5)) / F.lit(1000000000.0)

def normalize(column):
    return F.upper(F.regexp_replace(F.trim(column.cast('string')), '[^A-Za-z0-9]', ''))

def season_columns(df, date_col):
    month = F.month(date_col)
    return df.withColumn('season', F.when(month.between(6, 10), 'KHARIF').when(month.isin(11, 12, 1, 2, 3, 4), 'RABI')).withColumn('year', F.year(date_col) - F.when(month <= 4, 1).otherwise(0)).filter(F.col('season').isNotNull())

def season_bounds(df):
    return df.withColumn('season_start', F.to_date(F.concat_ws('-', 'year', F.when(F.col('season') == 'KHARIF', F.lit('06-01')).otherwise(F.lit('11-01'))))).withColumn('season_end', F.to_date(F.concat_ws('-', F.col('year') + F.when(F.col('season') == 'RABI', 1).otherwise(0), F.when(F.col('season') == 'KHARIF', F.lit('10-31')).otherwise(F.lit('04-30'))))).withColumn('weather_expected_days', F.datediff('season_end', 'season_start') + 1)

def normalize(column):
    return F.upper(F.regexp_replace(F.trim(column.cast('string')), '[^A-Za-z0-9]', ''))


In [ ]:
master = runtime.read_snapshot('source_master')
safe = [re.sub('[^A-Za-z0-9_]', '_', c).strip('_') for c in master.columns]
assert len(safe) == len(set(safe)), 'Column normalization collision'
master = master.toDF(*safe)
(
    require(master,
         ['district_censuscode',
             'district',
             'state',
             'year',
             'season',
             'centroid_lat',
             'centroid_lon',
             'coverage_sum_insured_total'])
)
base_columns = (
    [c for c in master.columns if not c.startswith(('weather_',
             'ndvi_',
             'evi_',
             'modis_')) and c not in ['model_ready_after_external_features',
             'feature_schema_version',
             'source_weather_available',
             'source_vegetation_available',
             'satellite_join_status',
             'weather_join_status',
             'grid_lat',
             'grid_lon',
             'coverage_insured_per_policy_row',
             'coverage_policy_rows_prior_zscore',
             'coverage_insured_prior_zscore']]
)
master = (
    master.select(*base_columns)
    .withColumn('year',
         F.col('year').cast('int'))
    .withColumn('district_censuscode',
         F.col('district_censuscode').cast('long'))
    .withColumn('season',
         F.upper(F.trim('season')))
    .withColumn('state_key',
         normalize(F.col('state')))
    .withColumn('district_key',
         normalize(F.col('district')))
    .withColumnRenamed('coverage_n_policy_rows',
         'coverage_n_source_rows')
    .withColumn('context_key',
         F.sha2(F.concat_ws('|',
             'state_key',
             'district_key',
             F.coalesce(F.col('district_censuscode').cast('string'), F.lit('unresolved')),
             'year',
             'season'),
             256))
)
unique(master, ['context_key'], 'Duplicate master contexts')
(
    unique(master.filter('district_censuscode is not null'),
         ['district_censuscode',
             'year',
             'season'],
         'Resolved key not unique')
)
master_n = master.count()
daily = runtime.read_snapshot('nasa_daily')
(
    require(daily,
         ['grid_lat',
             'grid_lon',
             'observation_date',
             'temp_mean_c',
             'precip_mm',
             'humidity_mean_pct'])
)
daily = (
    daily.select(F.col('grid_lat').cast('double'),
         F.col('grid_lon').cast('double'),
         F.to_date('observation_date').alias('observation_date'),
         *[F.col(c).cast('double').alias(c) for c in ['temp_mean_c',
             'precip_mm',
             'humidity_mean_pct']])
)
daily = (
    daily.filter('grid_lat between -90 and 90 and grid_lon between -180 and 180 and observation_date is not null')
)
for c, low, high in [('temp_mean_c', -90, 65), ('precip_mm', 0, 3000), ('humidity_mean_pct', 0, 100)]:
    daily = daily.withColumn(c, F.when(F.col(c).between(low, high) & ~F.isnan(c), F.col(c)))
(
    unique(daily,
         ['grid_lat',
             'grid_lon',
             'observation_date'],
         'Duplicate NASA daily keys; resolve at source')
)
daily = publish(daily, 'weather_daily', ['grid_lat'])
grids = daily.select('grid_lat', 'grid_lon').distinct()
points = (
    master.select('centroid_lat',
         'centroid_lon')
    .distinct()
    .filter('centroid_lat between -90 and 90 and centroid_lon between -180 and 180')
)
cross = points.crossJoin(F.broadcast(grids))
a = (
    F.pow(F.sin(F.radians(F.col('grid_lat') - F.col('centroid_lat')) / 2),
         2) + F.cos(F.radians('centroid_lat')) * F.cos(F.radians('grid_lat')) * F.pow(F.sin(F.radians(F.col('grid_lon') - F.col('centroid_lon')) / 2),
         2)
)
cross = (
    cross.withColumn('weather_grid_distance_km',
         2 * 6371.0088 * F.asin(F.sqrt(F.least(F.lit(1.0), a))))
)
nearest = (
    cross.withColumn('_rank',
         F.row_number().over(Window.partitionBy('centroid_lat',
             'centroid_lon').orderBy('weather_grid_distance_km',
             'grid_lat',
             'grid_lon')))
    .filter('_rank=1')
    .drop('_rank')
    .withColumn('weather_grid_accepted',
         F.col('weather_grid_distance_km') <= MAX_GRID_DISTANCE_KM)
    .withColumn('weather_grid_method',
         F.lit('nearest_available_NASA_request_location_haversine'))
)
nearest = publish(nearest, 'weather_grid_mapping')
seasons = (
    spark.createDataFrame([(y,
             s) for y in range(2018,
             2023) for s in ['KHARIF',
             'RABI']],
         ['year',
             'season'])
)
calendar = (
    season_bounds(grids.crossJoin(seasons))
    .withColumn('observation_date',
         F.explode(F.sequence('season_start',
             'season_end')))
)
aligned = calendar.join(daily, ['grid_lat', 'grid_lon', 'observation_date'], 'left')
keys = ['grid_lat', 'grid_lon', 'year', 'season']
order = Window.partitionBy(*keys).orderBy('observation_date')
aligned = (
    aligned.withColumn('_dry',
         F.col('precip_mm') < 1)
    .withColumn('_spell',
         F.sum(F.when(F.col('_dry'),
             0).otherwise(1)).over(order))
    .withColumn('_p3',
         F.when(F.count('precip_mm').over(order.rowsBetween(-2, 0)) == 3,
             F.sum('precip_mm').over(order.rowsBetween(-2, 0))))
)
spells = (
    aligned.groupBy(*keys + ['_spell'])
    .agg(F.sum(F.when(F.col('_dry'),
             1).otherwise(0)).alias('_dry_length'))
)
spells = spells.groupBy(*keys).agg(F.max('_dry_length').alias('weather_max_consecutive_dry_days'))
weather = (
    aligned.groupBy(*keys)
    .agg(F.first('season_end').alias('season_end'),
         F.first('weather_expected_days').alias('weather_expected_days'),
         F.count('temp_mean_c').alias('weather_temp_valid_days'),
         F.count('precip_mm').alias('weather_precip_valid_days'),
         F.count('humidity_mean_pct').alias('weather_humidity_valid_days'),
         F.avg('temp_mean_c').alias('weather_temp_mean_c'),
         F.sum('precip_mm').alias('weather_precip_total_mm'),
         F.avg('humidity_mean_pct').alias('weather_humidity_mean_pct'),
         F.max('temp_mean_c').alias('weather_temp_max_daily_mean_c'),
         F.sum(F.when(F.col('precip_mm').isNotNull(),
             F.when(F.col('precip_mm') < 1, 1).otherwise(0))).alias('weather_dry_days_lt_1mm'),
         F.sum(F.when(F.col('temp_mean_c').isNotNull(),
             F.when(F.col('temp_mean_c') >= 35, 1).otherwise(0))).alias('weather_heat_days_mean_ge_35c'),
         F.sum(F.when(F.col('precip_mm').isNotNull(),
             F.when(F.col('precip_mm') >= 20, 1).otherwise(0))).alias('weather_heavy_rain_days_ge_20mm'),
         F.sum(F.when(F.col('precip_mm').isNotNull(),
             F.when(F.col('precip_mm') >= 50, 1).otherwise(0))).alias('weather_heavy_rain_days_ge_50mm'),
         F.max('precip_mm').alias('weather_max_daily_precip_mm'),
         F.max('_p3').alias('weather_max_3day_precip_mm'),
         F.stddev_pop('precip_mm').alias('weather_precip_sd_mm'))
)
weather = (
    weather.join(spells,
         keys,
         'left')
    .withColumn('weather_valid_day_share',
         F.least('weather_temp_valid_days',
             'weather_precip_valid_days') / F.col('weather_expected_days'))
    .withColumn('weather_ready',
         F.col('weather_valid_day_share') >= MIN_DAY_SHARE)
    .withColumn('_valid_rain',
         F.when(F.col('weather_precip_valid_days') / F.col('weather_expected_days') >= MIN_DAY_SHARE,
             F.col('weather_precip_total_mm')))
)
history = (
    Window.partitionBy('grid_lat',
         'grid_lon',
         'season')
    .orderBy('year')
    .rowsBetween(Window.unboundedPreceding,
         -1)
)
weather = (
    weather.withColumn('weather_history_years',
         F.count('_valid_rain').over(history))
    .withColumn('weather_prior_precip_mm',
         F.avg('_valid_rain').over(history))
    .withColumn('weather_precip_anomaly_pct',
         F.when((F.col('weather_history_years') >= 2) & (F.col('weather_prior_precip_mm') > 0),
             100 * (F.col('weather_precip_total_mm') / F.col('weather_prior_precip_mm') - 1)))
    .drop('_valid_rain')
)
weather = publish(weather, 'weather_seasonal')
modis = runtime.read_snapshot('modis_seasonal')
require(modis, ['state', 'district', 'year', 'season', 'ndvi_mean', 'evi_mean'])
modis = (
    modis.select(normalize(F.col('state')).alias('state_key'),
         normalize(F.col('district')).alias('district_key'),
         F.col('year').cast('int'),
         F.upper(F.trim('season')).alias('season'),
         F.col('ndvi_mean').cast('double'),
         F.col('evi_mean').cast('double'))
    .distinct()
)
(
    unique(modis,
         ['state_key',
             'district_key',
             'year',
             'season'],
         'Conflicting MODIS keys; do not arbitrarily deduplicate')
)
for c in ['ndvi_mean', 'evi_mean']:
    modis = modis.withColumn(c, F.when(F.col(c).between(-1, 1) & ~F.isnan(c), F.col(c)))
mh = (
    Window.partitionBy('state_key',
         'district_key',
         'season')
    .orderBy('year')
    .rowsBetween(Window.unboundedPreceding,
         -1)
)
modis = (
    modis.withColumn('ndvi_history_years',
         F.count('ndvi_mean').over(mh))
    .withColumn('ndvi_prior_mean',
         F.avg('ndvi_mean').over(mh))
    .withColumn('ndvi_anomaly',
         F.when(F.col('ndvi_history_years') >= 2,
             F.col('ndvi_mean') - F.col('ndvi_prior_mean')))
    .withColumn('vegetation_source',
         F.lit('MODIS imported seasonal extract; QA metadata unavailable'))
)
modis = publish(modis, 'vegetation_seasonal')
gold = (
    master.join(nearest,
         ['centroid_lat',
             'centroid_lon'],
         'left')
    .join(weather,
         keys,
         'left')
    .join(modis,
         ['state_key',
             'district_key',
             'year',
             'season'],
         'left')
)
gh = (
    Window.partitionBy('district_censuscode',
         'season')
    .orderBy('year')
    .rowsBetween(Window.unboundedPreceding,
         -1)
)
gold = (
    gold.withColumn('_prior_insured_n',
         F.count('coverage_sum_insured_total').over(gh))
    .withColumn('_prior_insured_mean',
         F.avg('coverage_sum_insured_total').over(gh))
    .withColumn('_prior_insured_sd',
         F.stddev_samp('coverage_sum_insured_total').over(gh))
    .withColumn('coverage_insured_prior_zscore',
         F.when(F.col('district_censuscode').isNotNull() & (F.col('_prior_insured_n') >= 2) & (F.col('_prior_insured_sd') > 0),
             (F.col('coverage_sum_insured_total') - F.col('_prior_insured_mean')) / F.col('_prior_insured_sd')))
    .withColumn('source_vegetation_available',
         F.col('ndvi_mean').isNotNull() & F.col('evi_mean').isNotNull())
    .withColumn('source_weather_available',
         F.coalesce(F.col('weather_ready') & F.col('weather_grid_accepted'),
             F.lit(False)))
    .withColumn('satellite_join_status',
         F.when(F.col('source_vegetation_available'),
             'exact_normalized_name').otherwise('missing_or_invalid'))
    .withColumn('model_ready_after_external_features',
         F.coalesce(F.col('district_censuscode').isNotNull() & F.col('source_weather_available')
    & F.col('source_vegetation_available') & F.col('ndvi_anomaly').isNotNull() & F.col('weather_precip_anomaly_pct').isNotNull()
    & (F.col('coverage_sum_insured_total') > 0) & F.col('coverage_avg_premium_rate').isNotNull()
    & F.col('coverage_n_crops').isNotNull(),
             F.lit(False)))
    .withColumn('feature_schema_version',
         F.lit(VERSION))
    .withColumn('prediction_time_scope',
         F.lit('retrospective_post_season; publication_latency_not_verified'))
    .drop('_prior_insured_n',
         '_prior_insured_mean',
         '_prior_insured_sd')
)
unique(gold, ['context_key'], 'Gold fan-out')
assert gold.count() == master_n
gold = publish(gold, 'gold', ['year', 'season'])
coverage = (
    gold.groupBy('year',
         'season')
    .agg(F.count('*').alias('contexts'),
         F.sum(F.col('model_ready_after_external_features').cast('long')).alias('ready_contexts'),
         F.sum(F.col('source_weather_available').cast('long')).alias('weather_ready_contexts'),
         F.sum(F.col('source_vegetation_available').cast('long')).alias('vegetation_ready_contexts'))
)
publish(coverage, 'coverage')
(
    print('OBSERVED_STAGE_COMPLETE',
         json.dumps({'run_id': RUN_ID,
             'prefix': PREFIX,
             'inputs': SOURCE_VERSIONS,
             'rows': PUBLISHED}),
         flush=True)
)


Compare the rebuilt row counts and fields with `data/base_gold.csv.gz`, `data/weather_seasonal.csv.gz` and `data/vegetation_seasonal.csv.gz`. Numerical aggregation can differ across Spark runtimes. A successful rebuild is not a claim of bit-for-bit historical reproduction.